In [ ]:
# Rate designer: sample-based revenue neutrality
# R_sample = weighted sum of TOU-DR bills from our building sample
# Policy adjustment SHARES (wildfire, T&D, ROE) from utility GRC filing
# New rates = TOU-DR rates × scaling factor

import pandas as pd
import numpy as np
from itertools import product

# ============================================================================
# STEP 1: Compute R_sample from actual TOU-DR bills
# ============================================================================

bills_df = pd.read_csv('post_adoption_bills_sdge.csv')
BUILDING_WEIGHT = 252.3

R_SAMPLE = bills_df['tou_dr_bill'].dropna().sum() * BUILDING_WEIGHT
N_CARE = (bills_df['is_care'] == True).sum()
N_NONCARE = (bills_df['is_care'] == False).sum()

print(f"R_sample (weighted TOU-DR revenue): ${R_SAMPLE/1e9:.4f}B")
print(f"Sample: {len(bills_df)} buildings ({N_CARE} CARE, {N_NONCARE} non-CARE)")
print(f"  Weighted: {N_CARE * BUILDING_WEIGHT:,.0f} CARE, {N_NONCARE * BUILDING_WEIGHT:,.0f} non-CARE")

# ============================================================================
# STEP 2: Cost component SHARES from SDGE GRC filing
# ============================================================================

# Filed utility numbers (for computing shares only)
R_FILED = 1_561_695_600
R_TOTAL_UTILITY = 4_233_072_000
RES_SHARE = R_FILED / R_TOTAL_UTILITY

# Absolute costs from GRC filing (residential allocation)
WILDFIRE_COST = 413_873_000 * RES_SHARE    # $152.7M
TRANSMISSION_COST = 685_245_000 * RES_SHARE  # $252.8M
DISTRIBUTION_COST = 1_722_187_000 * RES_SHARE  # $635.4M
TD_COST = TRANSMISSION_COST + DISTRIBUTION_COST  # $888.2M
RATE_BASE = 13_590_538_000
EQUITY_SHARE = 0.52  # equity portion of capital structure

# Express as shares of filed revenue
WILDFIRE_SHARE = WILDFIRE_COST / R_FILED      # 9.78%
TD_SHARE = TD_COST / R_FILED                  # 56.87%
ROE_SHARE_PER_PP = (RATE_BASE * 0.01 * EQUITY_SHARE * RES_SHARE) / R_FILED  # 1.67%

print(f"\nCost shares from GRC filing:")
print(f"  Wildfire:  {WILDFIRE_SHARE*100:.2f}% of revenue (${WILDFIRE_COST/1e6:.1f}M filed)")
print(f"  T&D:       {TD_SHARE*100:.2f}% of revenue (${TD_COST/1e6:.1f}M filed)")
print(f"  ROE/pp:    {ROE_SHARE_PER_PP*100:.2f}% of revenue per 1pp reduction")

# TOU-DR rate structure (actual SDGE tariff)
TOU_DR_RATES = {
    'summer_peak': 0.60,
    'summer_midpeak': 0.527,
    'summer_offpeak': 0.45,
    'winter_peak': 0.58155,
    'winter_midpeak': 0.51899,
    'winter_offpeak': 0.50084
}

# Load actual TOU consumption weights
weights_df = pd.read_csv('tou_weights_sdge.csv')
TOU_WEIGHTS = dict(zip(weights_df['period'], weights_df['weight']))

print(f"\nTOU consumption weights:")
for period, weight in TOU_WEIGHTS.items():
    print(f"  {period:20s}: {weight*100:5.2f}%")

# ============================================================================
# STEP 3: Rate design function
# ============================================================================

def design_rate(fixed_pct_td=0, remove_wildfire=False, roe_reduction=0, care_fixed_ratio=0.4):
    """
    Design revenue-neutral rate using R_sample as baseline.
    
    Policy adjustments (wildfire, ROE) applied as shares of R_sample.
    Fixed charges allocated across sample CARE/non-CARE customers.
    TOU-DR rates scaled proportionally to hit volumetric revenue target.
    """
    # Revenue target = R_sample minus policy adjustments
    R_target = R_SAMPLE
    if remove_wildfire:
        R_target -= R_SAMPLE * WILDFIRE_SHARE
    if roe_reduction > 0:
        R_target -= R_SAMPLE * ROE_SHARE_PER_PP * roe_reduction
    
    # Fixed charge revenue = share of T&D costs
    R_fixed = R_SAMPLE * TD_SHARE * (fixed_pct_td / 100)
    
    # Per-customer fixed charges (using sample counts, weighted)
    N_care_w = N_CARE * BUILDING_WEIGHT
    N_noncare_w = N_NONCARE * BUILDING_WEIGHT
    total_weighted_customers = N_noncare_w + care_fixed_ratio * N_care_w
    fixed_non_care = R_fixed / total_weighted_customers / 12  # monthly
    fixed_care = fixed_non_care * care_fixed_ratio
    
    # Volumetric revenue
    R_vol = R_target - R_fixed
    
    # Scale TOU-DR rates to hit volumetric target
    scaling = R_vol / R_SAMPLE
    new_tou_rates = {k: v * scaling for k, v in TOU_DR_RATES.items()}
    
    # Weighted average for verification
    vol_avg = sum(new_tou_rates[p] * TOU_WEIGHTS[p] for p in new_tou_rates)
    
    return {
        'Scenario': f'F{fixed_pct_td}_WF{int(remove_wildfire)}_ROE{roe_reduction}',
        'Fixed_Pct_TD': fixed_pct_td,
        'Remove_Wildfire': remove_wildfire,
        'ROE_Reduction': roe_reduction,
        'Fixed_CARE': fixed_care,
        'Fixed_NonCARE': fixed_non_care,
        'Scaling': scaling,
        'Vol_Avg': vol_avg,
        'Total_Revenue': R_target,
        **new_tou_rates
    }

# ============================================================================
# GENERATE ALL SCENARIOS
# ============================================================================

print("\n" + "="*80)
print("SDGE RATE DESIGNER — Sample-Based Revenue Neutrality")
print("="*80)

fixed_percentages = [0, 25, 50, 75, 100]
wildfire_options = [False, True]
roe_reductions = [0, 0.5, 1.0, 1.5]

scenarios = []
for fixed_pct, wf, roe in product(fixed_percentages, wildfire_options, roe_reductions):
    scenarios.append(design_rate(fixed_pct_td=fixed_pct, remove_wildfire=wf, roe_reduction=roe))

df_scenarios = pd.DataFrame(scenarios)
df_scenarios = df_scenarios.round(4)
df_scenarios.to_csv('rate_scenarios_all_corrected.csv', index=False)

print(f"\nGenerated {len(df_scenarios)} rate scenarios")

# Verify revenue neutrality across fixed charge levels
print("\n" + "="*80)
print("REVENUE NEUTRALITY CHECK")
print("="*80)

baseline = df_scenarios[(df_scenarios['Remove_Wildfire'] == False) & 
                         (df_scenarios['ROE_Reduction'] == 0.0)]

print(f"\nBaseline scenarios (same R_target = R_sample = ${R_SAMPLE/1e9:.4f}B):")
for _, row in baseline.iterrows():
    print(f"  {row['Scenario']:15s}: scaling={row['Scaling']:.4f}  "
          f"FC_nonCARE=${row['Fixed_NonCARE']:.2f}/mo  "
          f"Vol avg=${row['Vol_Avg']:.4f}")

print(f"\nF0_WF0_ROE0 scaling = {baseline.iloc[0]['Scaling']:.4f} (should be 1.0000)")

# Show policy effects
print(f"\nPolicy effects on revenue target:")
for _, row in df_scenarios[df_scenarios['Fixed_Pct_TD'] == 0].iterrows():
    pct_reduction = (1 - row['Total_Revenue'] / R_SAMPLE) * 100
    print(f"  {row['Scenario']:20s}: ${row['Total_Revenue']/1e9:.4f}B  "
          f"({pct_reduction:+.2f}% vs baseline)")

print(f"\nSaved to: rate_scenarios_all_corrected.csv")

In [9]:
# SDGE data
RESIDENTIAL_REVENUE = 1_561_695_600
RESIDENTIAL_SALES_KWH = 4_810_000_000
CUSTOMERS = {'care': 372_135, 'non_care': 951_477, 'total': 1_323_612}
RATE_BASE = 13_590_538_000

res_share = RESIDENTIAL_REVENUE / 4_233_072_000
REVENUE_COMPONENTS = {
    'wildfire': 413_873_000 * res_share,
    'transmission': 685_245_000 * res_share,
    'distribution': 1_722_187_000 * res_share,
}
roe_impact = RATE_BASE * (0.5 / 100) * res_share
roe_impact*100/RESIDENTIAL_REVENUE

1.6052807511896796

In [14]:
# rate_designer.py - Revenue-neutral rate design with consumption-weighted TOU

import pandas as pd
import numpy as np
from itertools import product

# ============================================================================
# CONFIGURATION
# ============================================================================

# Actual SDGE residential data
RESIDENTIAL_REVENUE = 1_561_695_600  # From EIA
RESIDENTIAL_SALES_KWH = 4_810_000_000
CUSTOMERS = {'care': 372_135, 'non_care': 951_477, 'total': 1_323_612}

# Rate base
RATE_BASE = 13_590_538_000

# Revenue components (from table)
res_share = RESIDENTIAL_REVENUE / 4_233_072_000
REVENUE_COMPONENTS = {
    'wildfire': 413_873_000 * res_share,
    'transmission': 685_245_000 * res_share,
    'distribution': 1_722_187_000 * res_share,
}

# Current SDGE TOU-DR rate structure (from actual rates file)
BASELINE_TOU_RATES = {
    'summer_peak': 0.60,
    'summer_midpeak': 0.527,
    'summer_offpeak': 0.45,
    'winter_peak': 0.58155,
    'winter_midpeak': 0.51899,
    'winter_offpeak': 0.50084
}

# ============================================================================
# STEP 1: Calculate TOU consumption weights from building data
# ============================================================================

def get_tou_period(hour, month):
    """Determine TOU period for given hour and month."""
    is_summer = 6 <= month <= 10
    
    if is_summer:
        if 16 <= hour < 21:  # 4pm-9pm
            return 'peak'
        elif 6 <= hour < 16 or 21 <= hour < 22:
            return 'midpeak'
        else:
            return 'offpeak'
    else:
        if 16 <= hour < 21:
            return 'peak'
        elif 6 <= hour < 16 or 21 <= hour < 22:
            return 'midpeak'
        else:
            return 'offpeak'

def calculate_tou_weights_from_buildings(buildings_csv='baseline_bills_all_scenarios.csv', 
                                         buildings_dir='./Baseline_SDGE'):
    """
    Calculate consumption-weighted TOU distribution from actual buildings.
    Uses a sample of buildings to estimate the weights.
    """
    print("Calculating TOU consumption weights from building data...")
    
    # Load building list
    df = pd.read_csv(buildings_csv)
    sample_buildings = df['building_id'].head(100).tolist()  # Use 100 buildings for speed
    
    tou_consumption = {
        'summer_peak': 0, 'summer_midpeak': 0, 'summer_offpeak': 0,
        'winter_peak': 0, 'winter_midpeak': 0, 'winter_offpeak': 0
    }
    total_consumption = 0
    
    for i, building_id in enumerate(sample_buildings):
        try:
            # Load building data
            filepath = f"{buildings_dir}/{building_id}-0.parquet"
            building_df = pd.read_parquet(filepath)
            
            # Aggregate to hourly
            load_15min = building_df['out.electricity.total.energy_consumption'].values
            hourly = load_15min.reshape(-1, 4).sum(axis=1)
            
            # Assign to TOU periods
            for hour_idx, consumption in enumerate(hourly):
                month = (hour_idx // 730) + 1  # Approximate month
                hour = hour_idx % 24
                
                season = 'summer' if 6 <= month <= 10 else 'winter'
                period = get_tou_period(hour, month)
                
                tou_consumption[f'{season}_{period}'] += consumption
                total_consumption += consumption
            
            if (i+1) % 20 == 0:
                print(f"  Processed {i+1}/{len(sample_buildings)} buildings...")
                
        except Exception as e:
            print(f"  Error processing building {building_id}: {e}")
            continue
    
    # Convert to weights
    weights = {k: v/total_consumption for k, v in tou_consumption.items()}
    
    print("\nTOU Consumption Weights:")
    for period, weight in weights.items():
        print(f"  {period:20s}: {weight*100:5.2f}%")
    
    return weights

# Get TOU weights (calculate once or use defaults)
def get_tou_weights(calculate_from_buildings=False):
    """Get TOU consumption weights."""
    if calculate_from_buildings:
        return calculate_tou_weights_from_buildings()
    else:
        # Use approximate weights based on typical residential load
        return {
            'summer_peak': 0.12,      # Evening peak
            'summer_midpeak': 0.25,   # Daytime
            'summer_offpeak': 0.13,   # Night
            'winter_peak': 0.10,      # Evening peak
            'winter_midpeak': 0.28,   # Daytime
            'winter_offpeak': 0.12    # Night
        }

TOU_WEIGHTS = get_tou_weights(calculate_from_buildings=False)

# ============================================================================
# STEP 2: Design rates with revenue neutrality
# ============================================================================

def design_rate(fixed_pct_td=0, remove_wildfire=False, roe_reduction=0, care_fixed_ratio=0.4):
    """
    Design rate with exact revenue neutrality.
    
    All fixed charge scenarios (F0, F25, F50, F75) collect identical revenue.
    Policy changes (wildfire, ROE) reduce total revenue proportionally.
    """
    
    # Calculate target revenue
    revenue = RESIDENTIAL_REVENUE
    
    if remove_wildfire:
        revenue -= REVENUE_COMPONENTS['wildfire']
    
    if roe_reduction > 0:
        total_return_reduction = RATE_BASE * (roe_reduction / 100)
        roe_impact = total_return_reduction * res_share
        revenue -= roe_impact
    
    # Fixed charge design
    td_costs = REVENUE_COMPONENTS['transmission'] + REVENUE_COMPONENTS['distribution']
    fixed_revenue = td_costs * (fixed_pct_td / 100)
    
    # Income graduation
    total_weighted = (CUSTOMERS['care'] * care_fixed_ratio) + CUSTOMERS['non_care']
    fixed_non_care = fixed_revenue / total_weighted / 12  # monthly
    fixed_care = fixed_non_care * care_fixed_ratio
    
    # Volumetric revenue needed
    volumetric_revenue_needed = revenue - fixed_revenue
    
    # Target average rate (consumption-weighted)
    target_avg_rate = volumetric_revenue_needed / RESIDENTIAL_SALES_KWH
    
    # Calculate current weighted average
    current_weighted_avg = sum(BASELINE_TOU_RATES[period] * TOU_WEIGHTS[period] 
                               for period in BASELINE_TOU_RATES.keys())
    
    # Scale factor to achieve target
    scaling_factor = target_avg_rate / current_weighted_avg
    
    # Apply scaling to all TOU rates
    new_tou_rates = {k: v * scaling_factor for k, v in BASELINE_TOU_RATES.items()}
    
    # Verify revenue neutrality
    verification_avg = sum(new_tou_rates[period] * TOU_WEIGHTS[period] 
                          for period in new_tou_rates.keys())
    
    return {
        'Scenario': f'F{fixed_pct_td}_WF{int(remove_wildfire)}_ROE{roe_reduction}',
        'Fixed_Pct_TD': fixed_pct_td,
        'Remove_Wildfire': remove_wildfire,
        'ROE_Reduction': roe_reduction,
        'Fixed_CARE': fixed_care,
        'Fixed_NonCARE': fixed_non_care,
        'Vol_Avg': target_avg_rate,
        'Vol_Avg_Check': verification_avg,
        'Total_Revenue': revenue,
        **new_tou_rates
    }

# ============================================================================
# GENERATE ALL SCENARIOS
# ============================================================================

if __name__ == "__main__":
    print("="*80)
    print("SDGE RATE DESIGNER - Revenue Neutral")
    print("="*80)
    
    # Generate all combinations
    fixed_percentages = [0, 25, 50, 75, 100]
    wildfire_options = [False, True]
    roe_reductions = [0, 0.5, 1, 1.5]
    
    scenarios = []
    for fixed_pct, wf, roe in product(fixed_percentages, wildfire_options, roe_reductions):
        scenarios.append(design_rate(fixed_pct_td=fixed_pct, remove_wildfire=wf, roe_reduction=roe))
    
    df_scenarios = pd.DataFrame(scenarios)
    df_scenarios = df_scenarios.round(4)
    
    # Save
    df_scenarios.to_csv('rate_scenarios_all_corrected.csv', index=False)
    print(f"\nGenerated {len(df_scenarios)} rate scenarios")
    
    # Verify revenue neutrality
    print("\n" + "="*80)
    print("REVENUE NEUTRALITY CHECK")
    print("="*80)
    
    baseline_scenarios = df_scenarios[(df_scenarios['Remove_Wildfire'] == False) & 
                                      (df_scenarios['ROE_Reduction'] == 0)]
    
    print("\nBaseline scenarios (should have identical total revenue):")
    for _, row in baseline_scenarios.iterrows():
        print(f"  {row['Scenario']:15s}: ${row['Total_Revenue']/1e9:.4f}B  "
              f"(Vol avg: ${row['Vol_Avg']:.4f}, Check: ${row['Vol_Avg_Check']:.4f})")
    
    revenue_std = baseline_scenarios['Total_Revenue'].std()
    revenue_cv = revenue_std / baseline_scenarios['Total_Revenue'].mean() * 100
    
    print(f"\nCoefficient of Variation: {revenue_cv:.6f}%")
    print(f"(Should be ~0% for perfect revenue neutrality)")
    
    print("\n" + "="*80)
    print(f"Saved to: rate_scenarios_all_corrected.csv")
    print("="*80)

SDGE RATE DESIGNER - Revenue Neutral

Generated 40 rate scenarios

REVENUE NEUTRALITY CHECK

Baseline scenarios (should have identical total revenue):
  F0_WF0_ROE0    : $1.5617B  (Vol avg: $0.3247, Check: $0.3247)
  F25_WF0_ROE0   : $1.5617B  (Vol avg: $0.2785, Check: $0.2785)
  F50_WF0_ROE0   : $1.5617B  (Vol avg: $0.2324, Check: $0.2324)
  F75_WF0_ROE0   : $1.5617B  (Vol avg: $0.1862, Check: $0.1862)
  F100_WF0_ROE0  : $1.5617B  (Vol avg: $0.1400, Check: $0.1400)

Coefficient of Variation: 0.000000%
(Should be ~0% for perfect revenue neutrality)

Saved to: rate_scenarios_all_corrected.csv


In [13]:
import pandas as pd
metadata_df = pd.read_parquet('CA_Baseline_metadata_rescaled_twoincomes_puma20.parquet').reset_index()
print([c for c in metadata_df.columns if 'scale' in c.lower() or 'weight' in c.lower() or 'factor' in c.lower()])
print(metadata_df.head(3))


['weight', 'scaling_factor', 'scaled_out.electricity.total.energy_consumption.kwh']
   index  upgrade      weight  applicability  in.sqft  \
0      0        0  252.301639           True     1228   
1      1        0  252.301639           True     1228   
2      2        0  252.301639           True      623   

   in.representative_income     in.ahs_region in.aiannh_area  \
0                       0.0  Non-CBSA Pacific             No   
1                   66973.0  Non-CBSA Pacific             No   
2                   74877.0  Non-CBSA Pacific             No   

  in.area_median_income in.ashrae_iecc_climate_zone_2004  ... PUMA20_Pop20  \
0         Not Available                               4C  ...       136463   
1              100-120%                               4C  ...       136463   
2              120-150%                               3C  ...       159764   

  Part_Pop20 pPUMA10_Pop20 pPUMA20_Pop20   PUMA10_Land   PUMA20_Land  \
0     136463         100.0         100.0  9.2